# 예제 09 보충. 문서 작성 에이전트 들여다보기
### 예제 09를 마친 뒤 · 2일 차 · 20분

> **실행 방법**: 위에서부터 순서대로 실행하세요 (`Shift + Enter`). 이 노트북은 **Day2 폴더 안에** 두고 엽니다.

예제 09에서 `agent.run("ex09", ...)` 한 줄로 만든 Word 문서가 **안에서 어떻게 만들어지는지** 단계별로 확인합니다.

## 0. 실습 준비

FinanceAgent를 불러오고, 이 PC의 로컬 AI(Ollama)가 켜져 있는지 확인합니다.
`AI 연결 확인: qwen3:4b-instruct 준비 완료`가 나오면 성공입니다.

`DATA`는 **실습 데이터 파일의 위치**(`data/bank_customers.csv`, 1일 차 고객 데이터 45,211건)를 담아 둔 이름입니다. 아래 셀에서 `DATA`라고 쓰면 이 파일을 뜻합니다. 다른 파일로 분석하려면 이 줄의 파일 이름만 바꿉니다.

In [1]:
import json, urllib.request
from pathlib import Path
from finance_agent import FinanceAgent
import llm_config

DATA = "data/bank_customers.csv"      # 실습 데이터 파일 위치
models = [m["name"] for m in json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/tags"))["models"]]
print("데이터 파일:", Path(DATA).is_file())
print("AI 연결 확인:", llm_config.OLLAMA_MODEL, "준비 완료" if llm_config.OLLAMA_MODEL in models else "→ 모델이 없습니다. 강사에게 알려 주세요")
agent = FinanceAgent(provider="ollama")

from exercises import ex09_report


데이터 파일: True
AI 연결 확인: qwen3:4b-instruct 준비 완료


## 09-8. 📖 문서 작성 에이전트 들여다보기

지금까지 `agent.run("ex09", ...)` 한 줄로 Word 문서를 만들었습니다. 이 한 줄 안에서는 아래 **4단계**가 차례로 실행됩니다.
4단계 틀은 예제 06~10이 모두 같고, 예제마다 **파일 `exercises/ex09_report.py` 안의 함수만** 바뀝니다.

```
[① check]        →  [② function]          →  [③ prompt + AI]          →  [④ result]
 파일이 있나?          Pandas로 계산               계산 결과로 프롬프트를        AI 문장을 장별로 나누고
 CSV인가?              run_function()              만들어 AI에게 보냄             Word 문서로 조립
                       └ highlights()              make_prompt()                 show_result()
                                                   (보고서 / 기안문 2종)          ├ split_sections()
                                                                                 ├ build_report() / build_memo()
                                                                                 ├ make_chart_files()
                                                                                 └ check_numbers()
```

| 함수 · 설정 | 하는 일 | 누가 |
|---|---|---|
| `run_function(DATA, 질문)` | 고객 데이터를 읽어 기준 5개(연령대 · 잔액구간 · 대출상태 · 과거캠페인 · 직업군)별 가입률 표를 계산 | 🐍 Python |
| `highlights(표)` | 기준마다 가입률 **최고 · 최저** 고객군을 뽑음 (AI가 숫자를 고르지 않게) | 🐍 Python |
| `make_prompt(질문, 요약)` | `DOC_TYPE`에 따라 보고서용 · 기안문용 프롬프트를 만듦. 대괄호 제목(`[주요 현황]` 등)을 꼭 쓰게 지시 | 🐍 Python |
| (AI 호출) | 프롬프트를 받아 장별 문장을 씀 | 🤖 AI |
| `split_sections(AI 답, 제목 목록)` | AI 답을 대괄호 제목 기준으로 잘라 `{제목: 내용}`으로 나눔 | 🐍 Python |
| `build_report()` / `build_memo()` | 정해진 Word 틀(제목 · 분석 목적 · 표 · 차트 · 유의사항 · 결재란)에 AI 문장을 끼워 넣어 저장 | 🐍 Python |
| `make_chart_files()` | 보고서에 넣을 차트 PNG · HTML 저장 | 🐍 Python |
| `check_numbers(AI 답, 요약)` | AI 문장 속 숫자 중 계산 결과에 없는 숫자를 찾아 "확인 필요" 표시 | 🐍 Python |
| `DOC_TYPE` · `REPORT_SECTIONS` · `CAUTIONS` | 문서 종류 · AI가 쓸 장 제목 · 유의사항 문구 (설정값) | 강사 · 담당자 |

> 💡 핵심: **AI는 장별 문장만 씁니다.** 문서 틀 · 표 · 차트 · 숫자 검증은 모두 Python이 만듭니다. 그래서 AI가 형식을 틀려도 문서는 깨지지 않습니다.

아래 셀들은 **실행만** 하면서 단계별로 속을 확인합니다.

### 09-8-1. ③ AI가 실제로 받는 프롬프트 보기

Python이 계산한 숫자가 프롬프트 안에 들어가는 것을 확인합니다. (AI를 부르지 않아 바로 끝납니다)

In [2]:
ex09_report.DOC_TYPE = "보고서"
request = "정기예금 가입률 향상을 위한 고객관리 방안 보고서를 작성해 주세요."

summary = ex09_report.run_function(DATA, request)        # ② Python 계산
prompt = ex09_report.make_prompt(request, summary)      # ③ 프롬프트 만들기
print(prompt)

당신은 저축은행 경영기획부의 업무보고서 작성 담당자입니다.
업무 요청: 정기예금 가입률 향상을 위한 고객관리 방안 보고서를 작성해 주세요.

분석 결과 (Pandas 계산, 이 숫자만 사용):
{
 "전체 고객수": 45211,
 "전체 가입자수": 5289,
 "전체 가입률(%)": 11.7,
 "고객군별 최고·최저": {
  "연령대": {
   "최고": "60대 이상 33.6% (1,784명)",
   "최저": "40대 9.1% (11,655명)"
  },
  "잔액구간": {
   "최고": "5,000 초과 15.5% (2,845명)",
   "최저": "마이너스 5.6% (3,766명)"
  },
  "대출상태": {
   "최고": "대출 없음 18.2% (17,204명)",
   "최저": "둘 다 6.1% (4,367명)"
  },
  "과거캠페인": {
   "최고": "이전 가입 64.7% (1,511명)",
   "최저": "이전 연락 없음 9.2% (36,959명)"
  },
  "직업군": {
   "최고": "학생 28.7% (938명)",
   "최저": "생산·현장직 7.3% (9,732명)"
  }
 }
}

아래 세 제목을 대괄호까지 그대로 쓰고, 제목마다 내용을 작성하세요.
[주요 현황]
(전체 가입률과 고객군별 특징을 보고서 문체로 3~4문장)
[검토 의견]
(가입률이 높은 고객군·낮은 고객군의 의미와 해석상 주의점, 글머리 3개)
[향후 추진 방향]
(정기예금 가입률 향상을 위한 고객관리 방안, 글머리 3개)

보고서 문체(~로 나타남, ~할 필요가 있음)로 쓰고, 데이터에 없는 숫자는 쓰지 마세요.
연령·직업만으로 특정 고객을 고르거나 배제하는 방안은 쓰지 마세요.


### 09-8-2. ④ AI 답이 Word의 어느 장으로 가는지 보기

에이전트를 한 번 더 실행하고(`show=False`: 화면 출력 없이), AI 답을 `split_sections`가 어떻게 자르는지 봅니다.

In [3]:
from exercises.common import split_sections

report_run = agent.run("ex09", DATA, request, show=False)
print("에이전트가 넘겨준 값:", list(report_run.keys()), "\n")

sections = split_sections(report_run["ai_text"], ex09_report.REPORT_SECTIONS)
for name, text in sections.items():
    print(f"■ [{name}] → Word '{name}' 장 · {len(text)}자")
    print("  ", text[:100].replace("\n", " "), "…\n")

에이전트가 넘겨준 값: ['data_file', 'question', 'provider', 'data_summary', 'prompt', 'ai_text', 'artifacts', 'view_text', 'status', 'seconds'] 

■ [주요 현황] → Word '주요 현황' 장 · 281자
   전체 고객 중 정기예금 가입률은 11.7%로, 전체 고객수 45,211명 중 5,289명이 가입한 상태입니다. 연령대별로는 60대 이상 고객이 가입률 33.6%로 가장 높고, 40 …

■ [검토 의견] → Word '검토 의견' 장 · 347자
   가입률이 높은 고객군(예: 60대 이상, 5,000원 초과, 대출 없음, 학생)은 기존 고객의 신뢰와 자산 관리 의식이 강하다는 점에서, 그들의 행동 패턴이 정기예금 가입에 긍정적 …

■ [향후 추진 방향] → Word '향후 추진 방향' 장 · 334자
   첫째, 가입률이 낮은 잔액구간(마이너스) 고객에게는 금융 상품의 기초 이해를 돕는 간편한 교육 콘텐츠를 제공하여, 자산 관리의 필요성과 정기예금의 장점을 직접적으로 전달할 계획입니 …



> 장 하나가 0자로 나오면 AI가 대괄호 제목을 지키지 않은 것입니다. 이때는 다시 실행하거나 요청을 짧게 바꿉니다. (교재 50쪽)

### 09-8-3. 완성된 Word 문서 — 장마다 누가 썼나

In [4]:
from docx import Document

WRITER = {"2. 주요 현황": "🤖 AI 문장 + 🐍 표 · 차트", "3. 검토 의견": "🤖 AI 문장", "4. 향후 추진 방향": "🤖 AI 문장"}
for p in Document("outputs/정기예금_가입률_향상_전략보고서.docx").paragraphs:
    if p.style.name.startswith(("Heading", "Title")):
        print(f"{p.text[:22]:<24}", WRITER.get(p.text, "🐍 Python (정해진 문구 · 계산 숫자)"))

정기예금 가입률 향상을 위한 고객관리 방   🐍 Python (정해진 문구 · 계산 숫자)
1. 분석 목적                 🐍 Python (정해진 문구 · 계산 숫자)
2. 주요 현황                 🤖 AI 문장 + 🐍 표 · 차트
3. 검토 의견                 🤖 AI 문장
4. 향후 추진 방향              🤖 AI 문장
5. 유의사항                  🐍 Python (정해진 문구 · 계산 숫자)


### 09-8-4. [선택] 🔧 Python이 쓰는 부분 바꿔 보기

유의사항은 AI가 아니라 `CAUTIONS` 목록에서 그대로 들어갑니다. 우리 회사 문구를 한 줄 더하고 다시 만든 뒤, 원래대로 되돌립니다.

In [ ]:
ex09_report.CAUTIONS.append("본 자료는 내부 검토용이며 외부 배포를 금함.")     # ← 우리 회사 문구로 바꿔 보세요
agent.run("ex09", DATA, request, show=False)
ex09_report.CAUTIONS.pop()                                                 # 원래대로

for p in Document("outputs/정기예금_가입률_향상_전략보고서.docx").paragraphs:
    if "배포" in p.text or "교육용 초안" in p.text:
        print("·", p.text)

---
수고하셨습니다. 결과 파일은 `outputs` 폴더에 있습니다.